##   Best Practices

    #   1. Be Specific in Exceptions

        # Bad - Catching everything
        try:
            data = process()
        except:
            pass  # Swallows ALL errors

        # Good - Be specific
        try:
            data = process()
        except ValueError as e:
            logger.error(f"Value error: {e}")
            data = default_value
        except ConnectionError as e:
            logger.error(f"Connection error: {e}")
            retry_connection()

    #   2. Don't Suppress Exceptions Silently
        # Bad - Silent failure
        try:
            result = risky_operation()
        except:
            pass  # No idea if it worked

        # Good - Log or handle appropriately
        try:
            result = risky_operation()
        except Exception as e:
            logger.error(f"Operation failed: {e}")
            result = fallback_value()
            # Or re-raise if can't handle
            # raise

    #   4. Clean Up Resources Properly
        # Bad - Resource leak
            def read_file(filename):
                file = open(filename)
                data = file.read()
                return data  # File never closed if exception!

            # Good - Use with statement
            def read_file(filename):
                with open(filename) as file:
                    return file.read()  # Auto-closes even with exceptions

            # Or try-finally
            def read_file(filename):
                file = None
                try:
                    file = open(filename)
                    return file.read()
                finally:
                    if file:
                        file.close()

## Real-World Example: API Client

In [1]:
import requests
import json
import time
import logging

logging.basicConfig(level=logging.INFO)

class APIClient:
    def __init__(self, base_url):
        self.base_url = base_url
        self.max_retries = 3

    def fetch_data(self, endpoint):
        for attempt in range(self.max_retries):
            try:
                response = requests.get(f"{self.base_url}/{endpoint}", timeout=5)
                response.raise_for_status()  # Raises HTTPError for bad status
                return response.json()

            except requests.exceptions.Timeout:
                logging.warning(f"Timeout on attempt {attempt + 1}")
                if attempt == self.max_retries - 1:
                    raise TimeoutError("API request timed out after retries")
                time.sleep(2 ** attempt)  # Exponential backoff

            except requests.exceptions.ConnectionError:
                logging.error("Network connection error")
                raise ConnectionError("Cannot connect to API")

            except requests.exceptions.HTTPError as e:
                if response.status_code == 404:
                    raise ValueError(f"Endpoint {endpoint} not found")
                elif response.status_code == 401:
                    raise PermissionError("Unauthorized access")
                else:
                    raise RuntimeError(f"HTTP error: {e}")

            except json.JSONDecodeError as e:
                logging.error(f"Invalid JSON response: {e}")
                raise ValueError("API returned invalid JSON")

            except Exception as e:
                logging.critical(f"Unexpected error: {e}")
                raise  # Re-raise unexpected errors

# Usage
client = APIClient("https://api.example.com")
try:
    data = client.fetch_data("users")
except TimeoutError as e:
    print(f"Service unavailable: {e}")
except ValueError as e:
    print(f"Invalid response: {e}")
except Exception as e:
    print(f"Unexpected error: {e}")
else:
    print(f"Data fetched successfully: {len(data)} records")

ERROR:root:Network connection error


Unexpected error: Cannot connect to API
